# Practical Multivariate Forecasting

This notebook walks through a **complete end-to-end multivariate forecasting
pipeline** and provides an honest comparison with univariate ARIMA models.

A key lesson from Portilla's course (and from forecasting practice in general)
is that **VAR does not always outperform univariate models**.  Adding more
variables can help — but only if the cross-series relationships are strong
enough to overcome the additional estimation uncertainty.

This notebook covers:
1. End-to-end VAR pipeline (load, test, difference, fit, forecast, invert)
2. Fitting univariate ARIMA models for the same series
3. Head-to-head accuracy comparison
4. When multivariate models add value — practical guidelines

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.api import VAR
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

In [ ]:
def adf_test(series, name=""):
    """Run ADF test, return (is_stationary, p_value)."""
    result = adfuller(series.dropna(), autolag="AIC")
    stat, pvalue = result[0], result[1]
    label = "Stationary" if pvalue < 0.05 else "Non-stationary"
    print(f"  {name:>20s}  ADF={stat:+.4f}  p={pvalue:.4f}  => {label}")
    return pvalue < 0.05


def eval_forecast(actual, predicted, name=""):
    """Compute and print MAE, RMSE, MAPE."""
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    print(f"  {name:>20s}  MAE={mae:10.2f}  RMSE={rmse:10.2f}  MAPE={mape:6.2f}%")
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape}

---
## 1. Load and Explore the Data

We use **M2 Money Stock** and **Personal Consumption Expenditures (PCE)** —
two macroeconomic series that economic theory suggests are related.

In [ ]:
m2 = pd.read_csv(DATA_DIR / "M2SLMoneyStock.csv", index_col="Date", parse_dates=True)
pce = pd.read_csv(DATA_DIR / "PCEPersonalSpending.csv", index_col="Date", parse_dates=True)

df = pd.concat([m2, pce], axis=1).dropna()
df.columns = ["M2", "PCE"]
df.index.freq = pd.infer_freq(df.index)

print(f"Shape: {df.shape}")
print(f"Period: {df.index[0].date()} to {df.index[-1].date()}")
print(f"Frequency: {df.index.freq}")
print()
df.describe().round(1)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(df["M2"], color="tab:blue")
axes[0].set_ylabel("Billions $")
axes[0].set_title("M2 Money Stock")

axes[1].plot(df["PCE"], color="tab:orange")
axes[1].set_ylabel("Billions $")
axes[1].set_title("Personal Consumption Expenditures")

plt.tight_layout()
plt.show()

print(f"Pearson correlation (levels): {df['M2'].corr(df['PCE']):.3f}")
print("High correlation in levels is expected — both trend upward over time.")
print("The more meaningful question is whether changes in one predict changes in the other.")

---
## 2. Stationarity Tests and Differencing

In [ ]:
print("Step 1: ADF tests on levels")
m2_stat = adf_test(df["M2"], "M2")
pce_stat = adf_test(df["PCE"], "PCE")

print()
print("Step 2: First-difference and re-test")
df_diff = df.diff().dropna()
df_diff.columns = ["dM2", "dPCE"]

adf_test(df_diff["dM2"], "dM2")
adf_test(df_diff["dPCE"], "dPCE")

print()
print("Both series are I(1) — first-differencing makes them stationary.")

In [ ]:
# Visualize the differenced series
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(df_diff["dM2"], color="tab:blue", linewidth=0.8)
axes[0].axhline(0, color="grey", linestyle="--", alpha=0.5)
axes[0].set_title("$\\Delta$ M2 (month-over-month change)")

axes[1].plot(df_diff["dPCE"], color="tab:orange", linewidth=0.8)
axes[1].axhline(0, color="grey", linestyle="--", alpha=0.5)
axes[1].set_title("$\\Delta$ PCE (month-over-month change)")

plt.tight_layout()
plt.show()

---
## 3. Train/Test Split

In [ ]:
n_test = 24  # hold out last 24 months

# Differenced data split
train_diff = df_diff.iloc[:-n_test]
test_diff = df_diff.iloc[-n_test:]

# Level data split (for inverse-differencing and univariate ARIMA)
train_level = df.iloc[:-(n_test)]  # up to one before the test diff period
test_level = df.iloc[-(n_test):]

print(f"Train (diff):  {len(train_diff)} months  ({train_diff.index[0].date()} to {train_diff.index[-1].date()})")
print(f"Test (diff):   {len(test_diff)} months  ({test_diff.index[0].date()} to {test_diff.index[-1].date()})")
print(f"Test (level):  {len(test_level)} months")

---
## 4. Granger Causality — Is VAR Even Worth Trying?

Before investing effort in a multivariate model, check whether the
cross-series relationships are statistically significant.  If there
is no Granger causality, VAR is unlikely to outperform univariate models.

In [ ]:
print("Does dM2 Granger-cause dPCE?")
print("=" * 50)
gc1 = grangercausalitytests(train_diff[["dPCE", "dM2"]], maxlag=6, verbose=True)

In [ ]:
print("Does dPCE Granger-cause dM2?")
print("=" * 50)
gc2 = grangercausalitytests(train_diff[["dM2", "dPCE"]], maxlag=6, verbose=True)

In [ ]:
print("Interpretation:")
print("- If p-values are significant (< 0.05), the cross-variable lags")
print("  contain useful information, and VAR may outperform univariate models.")
print("- If p-values are large, VAR is unlikely to improve over ARIMA.")
print()
print("Even with significant Granger causality, the improvement may be")
print("small in practice. Let us find out by comparing forecasts.")

---
## 5. Fit the VAR Model

In [ ]:
# Lag order selection
var_model = VAR(train_diff)
lag_selection = var_model.select_order(maxlags=15)
print(lag_selection.summary())

print(f"\nAIC selects p = {lag_selection.aic}")
print(f"BIC selects p = {lag_selection.bic}")

In [ ]:
# Fit with AIC-selected order
var_result = var_model.fit(maxlags=15, ic="aic")
print(f"Fitted VAR({var_result.k_ar})")
print(f"AIC: {var_result.aic:.2f}")
print(f"BIC: {var_result.bic:.2f}")

In [ ]:
# Residual diagnostics
from statsmodels.stats.stattools import durbin_watson

resid = var_result.resid
dw = durbin_watson(resid)

print("Durbin-Watson statistics (should be near 2.0):")
for col, stat in zip(resid.columns, dw):
    print(f"  {col}: {stat:.4f}")

print()
print(f"Residual correlation (dM2, dPCE): {resid['dM2'].corr(resid['dPCE']):.3f}")

In [ ]:
# VAR forecast on differenced scale
k_ar = var_result.k_ar
var_forecast_diff = var_result.forecast(train_diff.values[-k_ar:], steps=n_test)

var_fc_df = pd.DataFrame(
    var_forecast_diff,
    index=test_diff.index,
    columns=["dM2_var", "dPCE_var"],
)

# Inverse-difference to get level forecasts
last_m2 = df["M2"].iloc[-(n_test + 1)]
last_pce = df["PCE"].iloc[-(n_test + 1)]

var_m2_level = last_m2 + var_fc_df["dM2_var"].cumsum()
var_pce_level = last_pce + var_fc_df["dPCE_var"].cumsum()

print("VAR forecast (first 6 months, levels):")
pd.DataFrame({
    "M2_actual": test_level["M2"].values[:6],
    "M2_VAR": var_m2_level.values[:6],
    "PCE_actual": test_level["PCE"].values[:6],
    "PCE_VAR": var_pce_level.values[:6],
}, index=test_level.index[:6]).round(1)

---
## 6. Fit Univariate ARIMA Models for Comparison

In [ ]:
# Fit ARIMA for M2
# Since the data is I(1), we use ARIMA(p,1,q) on the levels
# Try a few orders and pick the best by AIC
best_aic_m2 = np.inf
best_order_m2 = None
best_model_m2 = None

for p in range(0, 5):
    for q in range(0, 3):
        try:
            model = ARIMA(train_level["M2"], order=(p, 1, q))
            result = model.fit()
            if result.aic < best_aic_m2:
                best_aic_m2 = result.aic
                best_order_m2 = (p, 1, q)
                best_model_m2 = result
        except Exception:
            pass

print(f"Best ARIMA for M2: {best_order_m2}, AIC={best_aic_m2:.2f}")

In [ ]:
# Fit ARIMA for PCE
best_aic_pce = np.inf
best_order_pce = None
best_model_pce = None

for p in range(0, 5):
    for q in range(0, 3):
        try:
            model = ARIMA(train_level["PCE"], order=(p, 1, q))
            result = model.fit()
            if result.aic < best_aic_pce:
                best_aic_pce = result.aic
                best_order_pce = (p, 1, q)
                best_model_pce = result
        except Exception:
            pass

print(f"Best ARIMA for PCE: {best_order_pce}, AIC={best_aic_pce:.2f}")

In [ ]:
# ARIMA forecasts
arima_m2_forecast = best_model_m2.forecast(steps=n_test)
arima_pce_forecast = best_model_pce.forecast(steps=n_test)

# Set the index to match test_level
arima_m2_forecast.index = test_level.index
arima_pce_forecast.index = test_level.index

print("ARIMA forecasts (first 6 months):")
pd.DataFrame({
    "M2_actual": test_level["M2"].values[:6],
    "M2_ARIMA": arima_m2_forecast.values[:6],
    "PCE_actual": test_level["PCE"].values[:6],
    "PCE_ARIMA": arima_pce_forecast.values[:6],
}, index=test_level.index[:6]).round(1)

---
## 7. Head-to-Head Comparison

In [ ]:
print("=" * 70)
print("FORECAST ACCURACY COMPARISON (level scale, 24-month test set)")
print("=" * 70)

results = []

# M2
print("\nM2 Money Stock:")
r1 = eval_forecast(test_level["M2"].values, var_m2_level.values, "VAR")
r1["Model"] = "VAR"; r1["Series"] = "M2"
results.append(r1)

r2 = eval_forecast(test_level["M2"].values, arima_m2_forecast.values, f"ARIMA{best_order_m2}")
r2["Model"] = f"ARIMA{best_order_m2}"; r2["Series"] = "M2"
results.append(r2)

# PCE
print("\nPersonal Consumption Expenditures:")
r3 = eval_forecast(test_level["PCE"].values, var_pce_level.values, "VAR")
r3["Model"] = "VAR"; r3["Series"] = "PCE"
results.append(r3)

r4 = eval_forecast(test_level["PCE"].values, arima_pce_forecast.values, f"ARIMA{best_order_pce}")
r4["Model"] = f"ARIMA{best_order_pce}"; r4["Series"] = "PCE"
results.append(r4)

comparison_df = pd.DataFrame(results)[["Series", "Model", "MAE", "RMSE", "MAPE"]]
print("\n" + "=" * 70)
print(comparison_df.to_string(index=False))

In [ ]:
# Visual comparison — M2
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# M2
axes[0].plot(test_level["M2"], label="Actual", color="black", linewidth=2)
axes[0].plot(var_m2_level, label="VAR", color="tab:blue", linestyle="--")
axes[0].plot(arima_m2_forecast, label=f"ARIMA{best_order_m2}", color="tab:red", linestyle=":")
axes[0].set_title("M2 Money Stock — Forecast Comparison")
axes[0].set_ylabel("Billions $")
axes[0].legend()

# PCE
axes[1].plot(test_level["PCE"], label="Actual", color="black", linewidth=2)
axes[1].plot(var_pce_level, label="VAR", color="tab:blue", linestyle="--")
axes[1].plot(arima_pce_forecast, label=f"ARIMA{best_order_pce}", color="tab:red", linestyle=":")
axes[1].set_title("PCE — Forecast Comparison")
axes[1].set_ylabel("Billions $")
axes[1].legend()

plt.suptitle("VAR vs Univariate ARIMA — 24-Month Forecast", fontsize=14)
plt.tight_layout()
plt.show()

---
## 8. Honest Assessment

As Portilla's course demonstrated (and as the FPP3 textbook notes),
**VAR does not always outperform univariate models**.  Here is why:

In [ ]:
print("Why VAR may NOT outperform univariate ARIMA:")
print("=" * 55)
print()
print("1. PARAMETER PROLIFERATION")
print("   VAR estimates K^2 * p coefficients per lag.")
print("   With limited data, many coefficients are estimated")
print("   with high uncertainty, adding noise to forecasts.")
print()
print("2. WEAK CROSS-VARIABLE EFFECTS")
print("   If dM2 and dPCE are only weakly related (or related")
print("   contemporaneously but not in a leading/lagging way),")
print("   the extra variables add noise without information.")
print()
print("3. DIFFERENCING REMOVES INFORMATION")
print("   After differencing, the series may behave nearly")
print("   like white noise, leaving little for VAR to exploit.")
print()
print("4. FORECAST HORIZON")
print("   VAR advantages (if any) are typically strongest at")
print("   short horizons (1-3 steps). At longer horizons,")
print("   both VAR and ARIMA converge to the unconditional mean.")

In [ ]:
# Compare accuracy by forecast horizon
horizons = [1, 3, 6, 12, 24]
horizon_results = []

for h in horizons:
    for name, actual_col, var_fc, arima_fc in [
        ("M2", "M2", var_m2_level, arima_m2_forecast),
        ("PCE", "PCE", var_pce_level, arima_pce_forecast),
    ]:
        actual_h = test_level[actual_col].values[:h]
        rmse_var = np.sqrt(mean_squared_error(actual_h, var_fc.values[:h]))
        rmse_arima = np.sqrt(mean_squared_error(actual_h, arima_fc.values[:h]))
        horizon_results.append({
            "Series": name, "Horizon": h,
            "RMSE_VAR": round(rmse_var, 2),
            "RMSE_ARIMA": round(rmse_arima, 2),
            "VAR_better": rmse_var < rmse_arima,
        })

horizon_df = pd.DataFrame(horizon_results)
print("RMSE by Forecast Horizon:")
print(horizon_df.to_string(index=False))
print()
print("Look at the 'VAR_better' column — does VAR consistently win?")

---
## 9. When Do Multivariate Models Add Value?

Based on both theory and practical experience, VAR is most likely
to outperform univariate models when:

### Strong Granger causality
The cross-variable relationships must be statistically and economically
significant.  If $x$ does not Granger-cause $y$, including $x$ in a
VAR for $y$ will only add noise.

### Leading/lagging behavior
VAR excels when one variable **leads** another — e.g., housing starts
lead employment, or yield curve changes predict GDP growth.  If the
relationship is purely contemporaneous, VAR cannot exploit it for
forecasting (since we do not know the future value of the other variable).

### Economic theory supports the relationship
Ad hoc variable selection often leads to overfitting.  Use domain
knowledge to select variables that theory says should be related.

### Sufficient data relative to parameters
A rule of thumb: you need at least $5 \times (K + K^2 p)$ observations
for reliable estimation.  With $K=3, p=4$, that is $5 \times 39 = 195$ observations.

In [ ]:
# Practical checklist
print("PRACTICAL CHECKLIST: Should I Use VAR?")
print("=" * 50)
print()
print("Step 1: Do I have strong theoretical reasons to believe")
print("        these variables are dynamically related?")
print("        [ ] Yes  [ ] No -> stick with univariate")
print()
print("Step 2: Is there statistically significant Granger causality?")
print("        [ ] Yes  [ ] No -> VAR unlikely to help")
print()
print("Step 3: Does one variable LEAD the other (not just correlate)?")
print("        [ ] Yes  [ ] No -> contemporaneous correlation")
print("                           cannot be exploited for forecasting")
print()
print("Step 4: Do I have enough data? (>= 5x the number of parameters)")
print("        [ ] Yes  [ ] No -> reduce K or p")
print()
print("Step 5: After fitting, does VAR actually beat univariate on a")
print("        held-out test set?")
print("        [ ] Yes -> use VAR")
print("        [ ] No  -> use the simpler univariate model")
print()
print("The most important step is #5 — always validate empirically.")

---
## 10. Complete Pipeline Summary

Here is the full VAR forecasting workflow in one place:

In [ ]:
print("COMPLETE VAR FORECASTING PIPELINE")
print("=" * 50)
print()
print("1. LOAD & EXPLORE")
print("   - Load all series into a single DataFrame")
print("   - Plot, check correlations, understand the domain")
print()
print("2. TEST STATIONARITY")
print("   - ADF test each series")
print("   - If non-stationary: test for cointegration (Johansen)")
print("     - If cointegrated -> consider VECM")
print("     - If not -> difference and proceed with VAR")
print()
print("3. SELECT LAG ORDER")
print("   - model.select_order(maxlags=...) -> compare AIC/BIC")
print()
print("4. FIT VAR")
print("   - result = model.fit(maxlags=..., ic='aic')")
print()
print("5. DIAGNOSTICS")
print("   - Check residuals: ACF, Durbin-Watson, Ljung-Box")
print("   - Granger causality tests")
print("   - Impulse response functions (for interpretation)")
print()
print("6. FORECAST")
print("   - result.forecast(last_k_ar_values, steps=h)")
print()
print("7. INVERSE-TRANSFORM")
print("   - If you differenced: cumsum + last known level")
print("   - If you log-transformed: exponentiate")
print()
print("8. EVALUATE")
print("   - Compare to univariate ARIMA on held-out test set")
print("   - Use the model that performs better out-of-sample")

---
## Summary

- **VAR is a powerful tool** for modelling interrelated time series,
  but it is not a magic bullet.  The added complexity of estimating
  cross-variable coefficients means VAR needs **strong relationships**
  and **sufficient data** to outperform simpler univariate models.

- **Always compare** VAR to univariate ARIMA on a held-out test set.
  If ARIMA wins, use ARIMA -- parsimony matters.

- **Granger causality** is a necessary (but not sufficient) condition
  for VAR to add forecasting value.

- **Inverse-differencing** is critical: if you forget to undo the
  differencing, your forecasts will be on the wrong scale.

- **Practical guideline**: use multivariate models when domain knowledge
  and statistical tests both support cross-variable dynamics.  Otherwise,
  keep it simple with univariate methods.

In [ ]:
print("Chapter 10 complete.")
print()
print("Key models covered:")
print("  - VAR(p):  the workhorse multivariate model")
print("  - VARMA(p,q): more flexible but rarely needed")
print("  - VECM:    for cointegrated series")
print()
print("Key diagnostic tools:")
print("  - Granger causality: does one variable help predict another?")
print("  - Impulse response functions: how do shocks propagate?")
print("  - Johansen test: are the series cointegrated?")
print()
print("Key lesson: multivariate != better. Always validate empirically.")